# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.PRODUITS = Set()
model.MOIS = Set()
model.USINES = Set()
model.REGION = Set()
model.PROCEDES = Set()
model.ARC1 = Set(dimen=3, initialize=lambda m: [(i0,i1,i2) for i0 in m.PRODUITS for i1 in m.REGION for i2 in m.MOIS])
model.ARC2 = Set(dimen=3, initialize=lambda m: [(i0,i1,i2) for i0 in m.PRODUITS for i1 in m.USINES for i2 in m.REGION])
model.ARC3 = Set(dimen=3, initialize=lambda m: [(i0,i1,i2) for i0 in m.PRODUITS for i1 in m.USINES for i2 in m.PROCEDES])
model.ARC4 = Set(dimen=5, initialize=lambda m: [(i0,i1,i2,i3,i4) for i0 in m.PRODUITS for i1 in m.MOIS for i2 in m.USINES for i3 in m.REGION for i4 in m.PROCEDES])
model.ARC5 = Set(dimen=2, initialize=lambda m: [(i0,i1) for i0 in m.PRODUITS for i1 in m.USINES])

## 🔹 Parameters

In [ ]:
model.Revenu = Param(model.PRODUITS, within=NonNegativeReals)
model.COUT_STOCK = Param(model.PRODUITS, within=NonNegativeReals)
model.JOURS_PROD = Param(model.MOIS, within=NonNegativeReals)
model.CAP = Param(model.REGION, within=NonNegativeReals)
model.DEMANDE = Param(model.PRODUITS, model.REGION, model.MOIS, within=NonNegativeReals)
model.COUT_TRANSPORT = Param(model.PRODUITS, model.USINES, model.REGION, within=NonNegativeReals)
model.CoutProd = Param(model.PRODUITS, model.USINES, model.PROCEDES, within=NonNegativeReals)
model.TAUX_PROD = Param(model.PRODUITS, model.USINES, model.PROCEDES, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.PRODUITS, model.MOIS, model.USINES, model.REGION, model.PROCEDES, domain=NonNegativeReals)
model.STOCK = Var(model.PRODUITS, model.USINES, domain=NonNegativeReals)
model.pr = Var(domain=NonNegativeReals)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/Philbrick_data.dat')

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for p in model.PRODUITS:
    for r in model.REGION:
        model.c_for_0.add(sum(sum(model.X[p, 'fevrier', u, r, pr] for pr in model.PROCEDES) - model.STOCK[p, u] for u in model.USINES) <= model.DEMANDE[p, r, 'fevrier'])
model.c_for_1 = ConstraintList()
for p in model.PRODUITS:
    for r in model.REGION:
        model.c_for_1.add(sum(sum(model.X[p, 'mars', u, r, pr] for pr in model.PROCEDES) + model.STOCK[p, u] for u in model.USINES) <= model.DEMANDE[p, r, 'mars'])
model.c_for_2 = ConstraintList()
for m in model.MOIS:
    for u in model.USINES:
        model.c_for_2.add(sum(sum(1 / model.TAUX_PROD[p, u, pr] * sum(model.X[p, m, u, r, pr] for r in model.REGION) for pr in model.PROCEDES) for p in model.PRODUITS) <= model.JOURS_PROD[m])
model.c_for_3 = ConstraintList()
for u in model.USINES:
    model.c_for_3.add(sum(model.STOCK[p, u] for p in model.PRODUITS) <= 1000.0)

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.Revenu[p] * sum(sum(sum(sum(model.X[p, m, u, r, pr] for pr in model.PROCEDES) for r in model.REGION) for u in model.USINES) for m in model.MOIS) for p in model.PRODUITS) - sum(model.COUT_TRANSPORT[p, u, r] * sum(sum(model.X[p, m, u, r, pr] for pr in model.PROCEDES) for m in model.MOIS) for p,u,r in model.ARC2) - sum(model.COUT_STOCK[p] * sum(model.STOCK[p, u] for u in model.USINES) for p in model.PRODUITS) - sum(model.CoutProd[p, u, pr] * sum(sum(model.X[p, m, u, r, pr] for r in model.REGION) for m in model.MOIS) for p,u,pr in model.ARC3), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')